# 01 — Data QC

Merged RSV-A/B GISAID set: counts, sequence length, ambiguous bases, collection date, and geography. Justifies the QC filter.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath("../src"))
import numpy as np, pandas as pd
import matplotlib.pyplot as plt, seaborn as sns
sns.set_theme(style="whitegrid")
pd.set_option("display.width", 120)
from rsv import io, composition


In [ ]:
seqs = io.read_fasta("../data/full/RSV_full.fasta")
meta = io.load_metadata("../data/full/RSV_full_metadata.tsv")
print("sequences:", len(seqs), "| metadata rows:", len(meta))
meta[["Subtype","year","region"]].describe(include="all")

### Sequence length by subtype

In [ ]:
meta["len"] = meta["Sequence Length"]
fig, ax = plt.subplots(figsize=(7,4))
for s,c in zip("AB",["#2c7fb8","#de2d26"]):
    meta.loc[meta.Subtype==s,"len"].plot.hist(bins=120, alpha=.5, ax=ax, label=s, color=c)
ax.set_xlim(14800,15600); ax.axvline(14900,ls="--",c="k"); ax.axvline(15400,ls="--",c="k")
ax.set_xlabel("sequence length (bp)"); ax.legend(title="subtype"); plt.show()

### Apply the QC filter

In [ ]:
clean, report = io.qc_filter(meta, seqs)
pd.Series(report)

### Sequences per year and region (QC-passing)

In [ ]:
display(clean.groupby(["year"]).size().rename("n").to_frame().T)
clean.groupby(["region","Subtype"]).size().unstack(fill_value=0)